In [ ]:
import pandas as pd
import glob
files = glob.glob("data/raw/**/streamflow*.csv", recursive=True)

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(df.head())  # check it worked
print(df.shape)   # see total rows/columns

df.to_csv("data/raw/RunoffForcastingProject/combined_runoff.csv", index=False)

  NWM_version_number model_initialization_time model_output_valid_time  \
0               v2.1       2021-04-21_00:00:00     2021-04-21_01:00:00   
1               v2.1       2021-04-21_00:00:00     2021-04-21_02:00:00   
2               v2.1       2021-04-21_00:00:00     2021-04-21_03:00:00   
3               v2.1       2021-04-21_00:00:00     2021-04-21_04:00:00   
4               v2.1       2021-04-21_00:00:00     2021-04-21_05:00:00   

   streamflow_value  streamID  
0              0.45  20380357  
1              0.84  20380357  
2              1.51  20380357  
3              2.50  20380357  
4              3.80  20380357  
(652320, 5)


In [43]:
df1 = pd.read_csv("data/raw/RunoffForcastingProject/21609641/11266500_Strt_2021-04-20_EndAt_2023-04-21.csv")
df2 = pd.read_csv("data/raw/RunoffForcastingProject/20380357/09520500_Strt_2021-04-20_EndAt_2023-04-21.csv")
df1["streamID"] = 21609641
df2["streamID"] = 20380357

dfU = pd.concat([df1, df2], ignore_index=True)
print(dfU.head())  # check it worked
print(dfU.shape)   # see total rows/columns
dfU.to_csv("data/raw/RunoffForcastingProject/combined_ucgs.csv", index=False)

                    DateTime  USGSFlowValue 00060_cd  streamID USGS_GageID
0  2021-04-20 07:00:00+00:00          32.00        A  21609641         NaN
1  2021-04-20 07:15:00+00:00          32.56        A  21609641         NaN
2  2021-04-20 07:30:00+00:00          32.85        A  21609641         NaN
3  2021-04-20 07:45:00+00:00          33.41        A  21609641         NaN
4  2021-04-20 08:00:00+00:00          33.70        A  21609641         NaN
(135587, 5)


# Data Wrangling & Preprocessing (Josh Dula)
# Following the 6 data wrangling steps from zyBooks Ch. 5 (Table 5.1.1)
#
# References:
#   - zyBooks Ch. 5 (Sections 5.1, 5.2, 5.3, 5.4, 5.5)
#   - Han & Morrison (2022), Journal of Hydrology, 608, 127653
#   - Project Description: Improving NWM Forecasts Using DL Post-processing

In [ ]:
import pandas as pd
import numpy as np

nwm = pd.read_csv("data/raw/RunoffForcastingProject/combined_runoff.csv")
usgs = pd.read_csv("data/raw/RunoffForcastingProject/combined_ucgs.csv")

print("=== NWM Forecast Data ===")
print(nwm.info())
print(nwm.describe())
print(f"\nStations: {nwm['streamID'].unique()}")
print(f"Shape: {nwm.shape}")

print("\n=== USGS Observed Data ===")
print(usgs.info())
print(usgs.describe())
print(f"\nStations: {usgs['streamID'].unique()}")
print(f"Shape: {usgs.shape}")

print("\n=== USGS Column Issue ===")
print(f"Columns: {usgs.columns.tolist()}")
print(f"NaN counts:\n{usgs.isnull().sum()}")

In [ ]:
# fix column mismatch - both are quality codes ("A") but named differently
usgs["quality_cd"] = usgs["00060_cd"].fillna(usgs["USGS_GageID"])
usgs = usgs.drop(columns=["00060_cd", "USGS_GageID"])

print("Fixed USGS columns:", usgs.columns.tolist())
print(usgs.head())

# parse dates - NWM uses underscores, USGS has timezone
nwm["model_initialization_time"] = pd.to_datetime(
    nwm["model_initialization_time"], format="%Y-%m-%d_%H:%M:%S"
)
nwm["model_output_valid_time"] = pd.to_datetime(
    nwm["model_output_valid_time"], format="%Y-%m-%d_%H:%M:%S"
)

usgs["DateTime"] = pd.to_datetime(usgs["DateTime"], utc=True)
usgs["DateTime"] = usgs["DateTime"].dt.tz_localize(None)  # drop tz so they match NWM

print("\nDatetime dtypes after conversion:")
print(f"  NWM init:  {nwm['model_initialization_time'].dtype}")
print(f"  NWM valid: {nwm['model_output_valid_time'].dtype}")
print(f"  USGS time: {usgs['DateTime'].dtype}")

In [ ]:
# lead time = valid time - init time (in hours)
nwm["lead_time_hrs"] = (
    (nwm["model_output_valid_time"] - nwm["model_initialization_time"])
    .dt.total_seconds() / 3600
).astype(int)

print("NWM Lead Time Distribution:")
print(nwm["lead_time_hrs"].value_counts().sort_index())

nwm = nwm[nwm["lead_time_hrs"].between(1, 18)].copy()  # project only needs 1-18h
print(f"\nNWM after filtering to 1-18h: {nwm.shape}")

# resample USGS from 15-min to hourly (mean of 4 readings)
usgs = usgs.set_index("DateTime")
usgs_hourly = usgs.groupby("streamID")["USGSFlowValue"].resample("1h").mean().reset_index()
usgs_hourly.columns = ["streamID", "DateTime", "usgs_observed"]

print(f"\nUSGS after hourly resampling: {usgs_hourly.shape}")
print(usgs_hourly.head())

In [ ]:
print("Missing values before cleaning:")
print("NWM:\n", nwm.isnull().sum())
print("\nUSGS hourly:\n", usgs_hourly.isnull().sum())

nwm_dupes = nwm.duplicated(
    subset=["model_output_valid_time", "streamID", "lead_time_hrs"]
).sum()
usgs_dupes = usgs_hourly.duplicated(
    subset=["DateTime", "streamID"]
).sum()
print(f"\nNWM dupes: {nwm_dupes}")
print(f"USGS dupes: {usgs_dupes}")

nwm = nwm.drop_duplicates(
    subset=["model_output_valid_time", "streamID", "lead_time_hrs"],
    keep="first"
)
usgs_hourly = usgs_hourly.drop_duplicates(
    subset=["DateTime", "streamID"],
    keep="first"
)

usgs_hourly = usgs_hourly.dropna(subset=["usgs_observed"])
nwm = nwm.dropna(subset=["streamflow_value"])

print(f"\nAfter cleaning - NWM: {nwm.shape}, USGS: {usgs_hourly.shape}")

In [ ]:
# merge NWM + USGS on valid time and station
merged = nwm.merge(
    usgs_hourly,
    left_on=["model_output_valid_time", "streamID"],
    right_on=["DateTime", "streamID"],
    how="inner"
)

print(f"Merged shape: {merged.shape}")

# residual = forecast error, what the DL model will learn to predict
merged["residual"] = merged["streamflow_value"] - merged["usgs_observed"]

merged = merged[[
    "model_initialization_time", "model_output_valid_time",
    "streamID", "lead_time_hrs",
    "streamflow_value", "usgs_observed", "residual"
]].rename(columns={"streamflow_value": "nwm_forecast"})

print(f"Final columns: {merged.columns.tolist()}")
print(f"Shape: {merged.shape}")
print(merged.head(10))

In [ ]:
# train/val: Apr 2021 - Sep 2022, test: Oct 2022 - Apr 2023
train_val_end = pd.Timestamp("2022-09-30 23:59:59")
test_start = pd.Timestamp("2022-10-01 00:00:00")

train_val = merged[merged["model_output_valid_time"] <= train_val_end].copy()
test = merged[merged["model_output_valid_time"] >= test_start].copy()

print(f"Train/Val: {train_val.shape[0]} rows")
print(f"  {train_val['model_output_valid_time'].min()} to {train_val['model_output_valid_time'].max()}")
print(f"Test: {test.shape[0]} rows")
print(f"  {test['model_output_valid_time'].min()} to {test['model_output_valid_time'].max()}")

overlap = set(train_val.index) & set(test.index)
print(f"\nOverlap: {len(overlap)} rows (should be 0)")

print("\nPer-station counts:")
for name, subset in [("Train/Val", train_val), ("Test", test)]:
    counts = subset.groupby("streamID").size()
    print(f"  {name}: {dict(counts)}")

In [ ]:
import os
os.makedirs("data/processed", exist_ok=True)

train_val.to_csv("data/processed/train_val.csv", index=False)
test.to_csv("data/processed/test.csv", index=False)
merged.to_csv("data/processed/merged_all.csv", index=False)

print("Saved to data/processed/:")
print(f"  train_val.csv   ({train_val.shape[0]} rows)")
print(f"  test.csv        ({test.shape[0]} rows)")
print(f"  merged_all.csv  ({merged.shape[0]} rows)")
print(f"\nColumns: {merged.columns.tolist()}")